<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/07_RAG_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## RAG completo para consultas sobre Monotributo en ARCA

En este notebook integro las etapas desarrolladas en los notebooks anteriores para construir el sistema RAG completo.

Ya tengo:

- Un corpus de documentos oficiales de ARCA.
- El corpus corregido y normalizado.
- Los documentos divididos en chunks.
- Un embedding para cada chunk.
- Los embeddings almacenados en Pinecone.
- Una función de retrieval para recuperar los chunks relevantes.

En este notebook agrego el modelo de lenguaje de Hugging Face.

El flujo completo que implemento es:

Pregunta
↓
Embedding de la pregunta
↓
Retrieval en Pinecone
↓
Documentos relevantes
↓
Construcción del contexto
↓
Modelo de lenguaje
↓
Respuesta basada en el contexto

## Instalación de librerías

En esta celda instalo las librerías necesarias para conectarme con Pinecone, generar embeddings y utilizar el modelo de lenguaje de Hugging Face.

In [1]:
!pip install -q pinecone sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 11.0 MB/s eta 0:00:00


## Importación de librerías

En esta celda importo las librerías que voy a utilizar para construir el RAG.

In [2]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from google.colab import userdata

## Carga de las API keys

En esta celda recupero las API keys desde los Secrets de Google Colab.

No escribo las claves directamente en el notebook porque el proyecto será versionado en GitHub.

Utilizo Pinecone para el almacenamiento y retrieval vectorial y Hugging Face para acceder al modelo de lenguaje.

In [3]:
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
HF_TOKEN = userdata.get("HUGGINGFACE_TOKEN")

if not PINECONE_API_KEY:
    raise ValueError(
        "No se encontró PINECONE_API_KEY en los Secrets de Colab."
    )

if not HF_TOKEN:
    raise ValueError(
        "No se encontró HUGGINGFACE_TOKEN en los Secrets de Colab."
    )

print("API keys cargadas correctamente.")

API keys cargadas correctamente.


## Conexión con Pinecone

En esta celda me conecto con el índice de Pinecone que contiene los embeddings del corpus de ARCA.

No necesito montar Google Drive porque en esta etapa toda la información necesaria para el retrieval está almacenada en Pinecone.

In [4]:
PINECONE_INDEX_NAME = "arca-monotributo"

pc = Pinecone(
    api_key=PINECONE_API_KEY
)

index = pc.Index(
    PINECONE_INDEX_NAME
)

print("Conectado correctamente con Pinecone.")
print(f"Índice: {PINECONE_INDEX_NAME}")

Conectado correctamente con Pinecone.
Índice: arca-monotributo


## Verificación de la base vectorial

En esta celda verifico que Pinecone contenga los vectores que cargué anteriormente.

Espero encontrar aproximadamente 43 vectores, correspondientes a los chunks generados a partir del corpus inicial.

In [5]:
estadisticas = index.describe_index_stats()

print(estadisticas)

DescribeIndexStatsResponse(dimension=384, total_vector_count=43, metric='cosine', namespaces=1)


## Modelo de embeddings

En esta celda cargo el mismo modelo de embeddings que utilicé para representar los documentos.

Necesito utilizar el mismo modelo para la pregunta y para los documentos porque ambos deben estar representados en el mismo espacio vectorial.

In [6]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

modelo_embeddings = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Modelo de embeddings cargado.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado.


## Función de retrieval

En esta celda defino la función que transforma una pregunta en un embedding y consulta Pinecone.

Recupero los 5 chunks con mayor similitud semántica.

In [7]:
TOP_K = 5

def recuperar_documentos(pregunta):

    embedding_pregunta = modelo_embeddings.encode(
        pregunta
    ).tolist()

    resultado = index.query(
        vector=embedding_pregunta,
        top_k=TOP_K,
        include_metadata=True
    )

    return resultado.matches

## Construcción del contexto

En esta celda construyo el contexto que voy a entregar al modelo de lenguaje.

Tomo los textos de los documentos recuperados por Pinecone y los uno en un único texto.

Este contexto constituye la información documental que el modelo deberá utilizar para responder.

In [8]:
def construir_contexto(matches):

    bloques = []

    for i, match in enumerate(matches, start=1):

        texto = match.metadata.get(
            "texto",
            ""
        ).strip()

        if texto:

            bloques.append(
                f"[Documento {i}]\n{texto}"
            )

    return "\n\n---\n\n".join(
        bloques
    )

## Modelo de lenguaje

En esta celda cargo el modelo de lenguaje de Hugging Face que utilizaré para generar las respuestas.

El modelo recibe la pregunta junto con el contexto recuperado desde Pinecone.

El modelo no realiza el retrieval. Su función es generar una respuesta a partir de la información que le proporciono.

In [9]:
MODELO_GENERATIVO = "Qwen/Qwen2.5-0.5B-Instruct"

print("Cargando modelo...")

generador = pipeline(
    "text-generation",
    model=MODELO_GENERATIVO,
    token=HF_TOKEN,
    device_map="auto"
)

print("Modelo cargado correctamente.")

Cargando modelo...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Modelo cargado correctamente.


## Construcción del prompt

En esta celda construyo el prompt que recibe el modelo de lenguaje.

Le indico explícitamente que debe utilizar únicamente la información recuperada desde los documentos.

También le indico qué debe responder cuando el contexto no contiene información suficiente.

In [10]:
def construir_prompt(
    pregunta,
    contexto
):

    return f"""Respondé la pregunta utilizando únicamente la información del contexto.

Reglas:
- Respondé en español.
- No inventes información.
- No agregues información externa al contexto.
- Conservá correctamente los términos técnicos.
- Respondé de forma clara y concisa.
- Si el contexto no contiene información suficiente para responder, decí:
"No tengo suficiente información en los documentos recuperados."

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""

## Generación de la respuesta

En esta celda creo la función principal de generación.

La función recibe la pregunta y el contexto recuperado y utiliza el modelo de Hugging Face para generar la respuesta.

In [11]:
def generar_respuesta(
    pregunta,
    contexto
):

    prompt = construir_prompt(
        pregunta,
        contexto
    )

    salida = generador(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )

    return salida[0][
        "generated_text"
    ].strip()

## Función RAG completa

En esta celda integro todo el pipeline.

Primero recupero los documentos relevantes desde Pinecone.

Después construyo el contexto.

Finalmente envío la pregunta y el contexto al modelo de lenguaje para generar la respuesta.

Esta función representa el sistema RAG completo.

In [12]:
def ejecutar_rag(pregunta):

    matches = recuperar_documentos(
        pregunta
    )

    contexto = construir_contexto(
        matches
    )

    respuesta = generar_respuesta(
        pregunta,
        contexto
    )

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "matches": matches,
        "contexto": contexto
    }

## Primera prueba del RAG

En esta celda pruebo el sistema con una pregunta sobre la Clave Fiscal.

Quiero comprobar el funcionamiento completo del pipeline y observar tanto los documentos recuperados como la respuesta generada.

In [13]:
pregunta = "¿Cómo puedo obtener la clave fiscal?"

resultado = ejecutar_rag(
    pregunta
)

print("=" * 80)
print("PREGUNTA")
print("=" * 80)

print(pregunta)

print("\n" + "=" * 80)
print("RESPUESTA RAG")
print("=" * 80)

print(
    resultado["respuesta"]
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


PREGUNTA
¿Cómo puedo obtener la clave fiscal?

RESPUESTA RAG
Obteniendo la clave fiscal desde la aplicación móvil "ARCA" es la única manera de obtenerla. 

No tengo suficiente información en los documentos recuperados. 

Respuesta: Obtener la clave fiscal desde la aplicación móvil "ARCA". 

No tengo suficiente información en los documentos recuperados. 

Respuesta: Obtener la clave fiscal desde la aplicación móvil "ARCA". 

No tengo suficiente información en los documentos recuperados. 

Respuesta: Obtener la clave fiscal desde la aplicación móvil "ARCA". 

No tengo suficiente información en los documentos recuperados. 

Respuesta: Obtener la clave fiscal desde la aplicación móvil "ARCA". 

No tengo suficiente información en los documentos recuperados.


## Inspección del retrieval antes de generar la respuesta

En esta celda realizo directamente una consulta a Pinecone para comprobar qué documentos recupera el sistema para una pregunta sobre la Clave Fiscal.

Genero el embedding de la pregunta utilizando el mismo modelo que utilicé para los documentos y consulto el índice de Pinecone.

Quiero verificar primero que el retrieval encuentra información relevante antes de analizar la generación de la respuesta del modelo de lenguaje.

In [14]:
pregunta = "¿Cómo puedo obtener la clave fiscal?"

# Generar embedding de la pregunta
embedding_pregunta = modelo_embeddings.encode(
    pregunta
).tolist()

# Consultar Pinecone
resultado_retrieval = index.query(
    vector=embedding_pregunta,
    top_k=5,
    include_metadata=True
)

matches = resultado_retrieval.matches

print("=" * 80)
print("DOCUMENTOS RECUPERADOS PARA LA PREGUNTA")
print("=" * 80)

for i, match in enumerate(
    matches,
    start=1
):

    metadata = match.metadata

    print(f"\nDocumento {i}")

    print(
        f"Score: {match.score:.4f}"
    )

    print(
        f"Archivo: "
        f"{metadata.get('archivo', 'desconocido')}"
    )

    print(
        f"Documento ID: "
        f"{metadata.get('documento_id', 'desconocido')}"
    )

    print(
        f"Chunk: "
        f"{metadata.get('chunk', 'desconocido')}"
    )

    print("\nTexto:")

    print(
        metadata.get(
            "texto",
            ""
        )
    )

    print("\n" + "-" * 80)

DOCUMENTOS RECUPERADOS PARA LA PREGUNTA

Documento 1
Score: 0.6147
Archivo: corpus_arca_monotributo.json
Documento ID: inicio
Chunk: 1

Texto:
egorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder después darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, hay que
ingresar con clave fiscal
y habilitar el
domicilio fiscal electrónico
, indispensable para recibir las comunicaciones oficiales de esta Agencia de Recaudación y
evitar estafas
.
Una vez realizadas estas gestiones, se debe acceder al
servicio con clave fiscal "Registro Único Tributario – RUT"
para ingresar la información de los
domicilios
, declarar las
actividades a desarrollar
y comenzar el proceso de alta en el monotributo.
Todos los trámites necesarios

--------------------------------------------------------

## Inspección del contexto

En esta celda construyo y visualizo el contexto que voy a entregar al modelo de lenguaje.

Quiero comprobar que el modelo recibe efectivamente la información relevante recuperada desde Pinecone antes de analizar la calidad de la respuesta generada.

In [15]:
contexto_prueba = construir_contexto(
    matches
)

print("=" * 80)
print("CONTEXTO QUE RECIBIRÁ EL MODELO")
print("=" * 80)

print(contexto_prueba)

CONTEXTO QUE RECIBIRÁ EL MODELO
[Documento 1]
egorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder después darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, hay que
ingresar con clave fiscal
y habilitar el
domicilio fiscal electrónico
, indispensable para recibir las comunicaciones oficiales de esta Agencia de Recaudación y
evitar estafas
.
Una vez realizadas estas gestiones, se debe acceder al
servicio con clave fiscal "Registro Único Tributario – RUT"
para ingresar la información de los
domicilios
, declarar las
actividades a desarrollar
y comenzar el proceso de alta en el monotributo.
Todos los trámites necesarios

---

[Documento 2]
Pagos
Recategorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general

## Inspección del prompt

En esta celda construyo el prompt completo que recibe el modelo de lenguaje.

Quiero verificar que la pregunta, el contexto y las instrucciones del sistema RAG estén correctamente organizados antes de generar la respuesta.

In [16]:
prompt_prueba = construir_prompt(
    pregunta,
    contexto_prueba
)

print("=" * 80)
print("PROMPT COMPLETO")
print("=" * 80)

print(prompt_prueba)

PROMPT COMPLETO
Respondé la pregunta utilizando únicamente la información del contexto.

Reglas:
- Respondé en español.
- No inventes información.
- No agregues información externa al contexto.
- Conservá correctamente los términos técnicos.
- Respondé de forma clara y concisa.
- Si el contexto no contiene información suficiente para responder, decí:
"No tengo suficiente información en los documentos recuperados."

CONTEXTO:
[Documento 1]
egorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder después darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, hay que
ingresar con clave fiscal
y habilitar el
domicilio fiscal electrónico
, indispensable para recibir las comunicaciones oficiales de esta Agencia de Recaudación y
evitar estafas
.
Una vez realizadas estas gestiones, se 

## Prueba de generación con un único documento relevante

En esta celda pruebo la generación utilizando únicamente el documento que contiene información específica sobre la obtención de la Clave Fiscal.

De esta manera elimino temporalmente el ruido producido por documentos menos relevantes y compruebo si el modelo puede generar correctamente una respuesta cuando recibe un contexto claro.

In [17]:
# Utilizo solamente el Documento 2 recuperado
contexto_controlado = matches[1].metadata["texto"]

pregunta_controlada = "¿Cómo puedo obtener la clave fiscal?"

respuesta_controlada = generar_respuesta(
    pregunta_controlada,
    contexto_controlado
)

print("=" * 80)
print("PRUEBA CON CONTEXTO CONTROLADO")
print("=" * 80)

print(respuesta_controlada)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PRUEBA CON CONTEXTO CONTROLADO
"obtener la clave fiscal desde la aplicación móvil 'ARCA' utilizando la opción 'Solicitar o recuperar la clave fiscal'. "
No tengo suficiente información en los documentos recuperados.
